# 🏃 Strava Performance Lab — Análise Exploratória Avançada
**Equipe**: Engenheiro de Dados · Treinador de Alta Performance · Fisiologista · Médico Esportivo


In [24]:
import sqlite3, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import folium
from folium.plugins import HeatMap
import polyline as pl
from pathlib import Path
from IPython.display import display

warnings.filterwarnings('ignore')

# ── FILTROS PARAMETRIZÁVEIS (altere aqui — integração futura com Streamlit) ──
DATE_START  = None          # ex: '2024-01-01'
DATE_END    = None          # ex: '2025-12-31'
SPORT_TYPES = None          # ex: ['Run', 'Ride'] ou None para todas

DB_PATH = Path('..') / 'data' / 'strava.db'
conn    = sqlite3.connect(DB_PATH)
print(f'✅ Conectado: {DB_PATH.resolve()}')

✅ Conectado: C:\Users\gabri\OneDrive\Belgeler\GitHub\strava-performance-analytics\data\strava.db


In [25]:
# ── CARGA E PRÉ-PROCESSAMENTO ─────────────────────────────────────────────────
def load_activities(conn, date_start=None, date_end=None, sport_types=None):
    q = 'SELECT * FROM activities WHERE 1=1'
    params = []
    if date_start:
        q += ' AND start_date_local >= ?'; params.append(date_start)
    if date_end:
        q += ' AND start_date_local <= ?'; params.append(date_end + 'T23:59:59')
    if sport_types:
        if sport_types:
            placeholders = ','.join(['?'] * len(sport_types))
            q += f' AND sport_type IN ({placeholders})'
            params.extend(sport_types)
    q += ' ORDER BY start_date_local'
    df = pd.read_sql(q, conn, params=params)

    df['start_date_local'] = pd.to_datetime(df['start_date_local'])
    df['date']      = df['start_date_local'].dt.date
    df['week']      = df['start_date_local'].dt.to_period('W')
    df['month']     = df['start_date_local'].dt.to_period('M')
    df['year']      = df['start_date_local'].dt.year
    df['dist_km']   = df['distance'] / 1000
    df['duration_h']= df['moving_time'] / 3600
    df['speed_kmh'] = np.where(df['moving_time'] > 0, df['distance'] / df['moving_time'] * 3.6, np.nan)
    df['pace_min_km']= np.where((df['moving_time'] > 0) & (df['dist_km'] > 0),
                                (df['moving_time'] / 60) / df['dist_km'], np.nan)
    # Custo cardíaco: BPM por unidade de velocidade (m/s) — cai com o tempo = mais eficiente
    df['cardiac_cost'] = df['average_heartrate'] / (df['distance'] / df['moving_time'].replace(0, np.nan))
    return df

df = load_activities(conn, DATE_START, DATE_END, SPORT_TYPES)
print(f'📊 {len(df)} atividades | {df["sport_type"].value_counts().to_dict()}')
df[['start_date_local','sport_type','dist_km','duration_h','average_heartrate']].tail()

📊 353 atividades | {'Walk': 144, 'WeightTraining': 72, 'Run': 70, 'Swim': 34, 'Ride': 15, 'Soccer': 10, 'Workout': 4, 'Racquetball': 1, 'Tennis': 1, 'RockClimbing': 1, 'Rowing': 1}


,start_date_local,sport_type,dist_km,duration_h,average_heartrate
348,2026-05-19 09:29:49+00:00,Walk,4.6070,1.618611,89.7
349,2026-05-20 12:10:59+00:00,Walk,2.5399,0.666667,93.3
350,2026-05-24 10:01:56+00:00,Walk,2.0024,0.534167,101.8
351,2026-05-25 20:08:07+00:00,WeightTraining,0.0000,0.477500,110.0
352,2026-05-25 20:37:31+00:00,Run,2.0300,0.262500,150.9


---
## 🏋️ Pilar 1 — Consistência e Carga de Treino (ACWR)
> **Fisiologia**: O ACWR (Acute:Chronic Workload Ratio) compara a carga aguda (7 dias) com a carga crônica (28 dias). ACWR > 1.3 = zona de alto risco de lesão por overtraining. ACWR < 0.8 = atleta sub-treinado (perda de adaptação).

In [26]:
# Volume semanal empilhado por modalidade
weekly = (
    df.groupby(['week', 'sport_type'])
    .agg(dist_km=('dist_km','sum'), duration_h=('duration_h','sum'), n=('id','count'))
    .reset_index()
)
weekly['week_str'] = weekly['week'].astype(str)

fig = px.bar(weekly, x='week_str', y='dist_km', color='sport_type', barmode='stack',
             title='Volume Semanal por Modalidade (km)',
             labels={'week_str':'Semana','dist_km':'Distância (km)'})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

In [27]:
# ACWR — Acute:Chronic Workload Ratio
daily = df.groupby('date')['dist_km'].sum().reset_index()
daily = daily.set_index('date')
daily.index = pd.to_datetime(daily.index)
daily = daily.asfreq('D', fill_value=0).reset_index()
daily.columns = ['date', 'dist_km']

daily['acute_7d']    = daily['dist_km'].rolling(7,  min_periods=1).mean()
daily['chronic_28d'] = daily['dist_km'].rolling(28, min_periods=1).mean()
daily['ACWR']        = daily['acute_7d'] / daily['chronic_28d'].replace(0, np.nan)

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_trace(go.Bar(x=daily['date'], y=daily['dist_km'], name='Carga Diária (km)', opacity=0.35, marker_color='steelblue'), secondary_y=False)
fig.add_trace(go.Scatter(x=daily['date'], y=daily['ACWR'], name='ACWR', line=dict(color='crimson', width=2)), secondary_y=True)
fig.add_hline(y=1.3, line_dash='dash', line_color='orange', annotation_text='Risco Lesão (>1.3)', secondary_y=True)
fig.add_hline(y=0.8, line_dash='dash', line_color='royalblue', annotation_text='Sub-treino (<0.8)', secondary_y=True)
fig.update_layout(title='ACWR — Monitoramento de Risco de Overtraining', xaxis_title='Data')
fig.update_yaxes(title_text='Distância (km)', secondary_y=False)
fig.update_yaxes(title_text='ACWR', secondary_y=True)
fig.show()

---
## ❤️ Pilar 2 — Evolução de Ritmo e Eficiência Cardíaca
> **Fisiologia**: O Custo Cardíaco (BPM / velocidade) mede o "preço" que seu coração paga por cada m/s percorrido. Com o treino adequado, ele deve CAIR ao longo dos meses — você corre mais rápido com menos esforço cardíaco. É um marcador de adaptação aeróbica.

In [28]:
df_run = df[df['sport_type'].isin(['Run','TrailRun','VirtualRun'])].copy()
df_run = df_run[df_run['pace_min_km'].between(3, 15) & df_run['average_heartrate'].notna()]

if df_run.empty:
    print('⚠️ Nenhuma corrida no período filtrado.')
else:
    # Scatter: pace ao longo do tempo, colorido por FC
    fig = px.scatter(
        df_run, x='start_date_local', y='pace_min_km',
        color='average_heartrate', size='dist_km', hover_data=['name','dist_km'],
        title='Evolução do Pace de Corrida — Colorido pela FC Média',
        labels={'start_date_local':'Data','pace_min_km':'Pace (min/km)','average_heartrate':'FC Média'},
        color_continuous_scale='RdYlGn_r'
    )
    fig.update_yaxes(autorange='reversed')  # pace menor = mais rápido
    fig.show()

In [29]:
if not df_run.empty:
    cc = df_run.groupby('month').agg(
        cardiac_cost=('cardiac_cost','mean'),
        pace_mean=('pace_min_km','mean'),
        n=('id','count')
    ).reset_index()
    cc['month_str'] = cc['month'].astype(str)

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Scatter(x=cc['month_str'], y=cc['cardiac_cost'],
                             name='Custo Cardíaco (bpm/ms⁻¹)', line=dict(color='crimson', width=2),
                             mode='lines+markers'), secondary_y=False)
    fig.add_trace(go.Scatter(x=cc['month_str'], y=cc['pace_mean'],
                             name='Pace Médio (min/km)', line=dict(color='royalblue', width=2, dash='dot'),
                             mode='lines+markers'), secondary_y=True)
    fig.update_layout(title='Eficiência Aeróbica Mensal — Custo Cardíaco vs Pace')
    fig.update_yaxes(title_text='Custo Cardíaco ↓ melhor', secondary_y=False)
    fig.update_yaxes(title_text='Pace min/km ↓ mais rápido', autorange='reversed', secondary_y=True)
    fig.update_layout(xaxis_tickangle=-45)
    fig.show()

---
## 🫀 Pilar 3 — Distribuição de Esforço por Zonas de FC
> **Fisiologia (Treino Polarizado)**: A distribuição ideal é ~80% do tempo em Z1-Z2 (aeróbico leve) e ~20% em Z4-Z5 (alta intensidade). Excesso de Z3 (zona cinzenta) causa fadiga crônica sem o benefício de estímulo de alta intensidade — é o erro mais comum em amadores.

In [30]:
q = '''
    SELECT z.activity_id, z.zone_type, z.zone_index, z.time_in_zone,
           a.start_date_local, a.sport_type
    FROM activity_zones z
    JOIN activities a ON z.activity_id = a.id
    WHERE z.zone_type = 'heartrate'
'''
df_z = pd.read_sql(q, conn)
df_z['start_date_local'] = pd.to_datetime(df_z['start_date_local'])
df_z['month'] = df_z['start_date_local'].dt.to_period('M')

if DATE_START: df_z = df_z[df_z['start_date_local'] >= pd.Timestamp(DATE_START)]
if DATE_END:   df_z = df_z[df_z['start_date_local'] <= pd.Timestamp(DATE_END)]
if SPORT_TYPES: df_z = df_z[df_z['sport_type'].isin(SPORT_TYPES)]

ZONE_LABELS = {1:'Z1 — Recuperação',2:'Z2 — Base Aeróbica',3:'Z3 — Limiar',4:'Z4 — Alta Intensidade',5:'Z5 — Máximo'}
ZONE_COLORS = {v: c for v,c in zip(ZONE_LABELS.values(), ['#4FC3F7','#81C784','#FFD54F','#FF8A65','#E53935'])}
df_z['zone_label'] = df_z['zone_index'].map(ZONE_LABELS)

zm = df_z.groupby(['month','zone_label'])['time_in_zone'].sum().reset_index()
zm['month_str'] = zm['month'].astype(str)
zm['pct'] = zm.groupby('month_str')['time_in_zone'].transform(lambda x: x / x.sum() * 100)

fig = px.bar(zm, x='month_str', y='pct', color='zone_label',
             color_discrete_map=ZONE_COLORS, barmode='stack',
             title='Distribuição Mensal do Tempo por Zona de FC (%)',
             labels={'month_str':'Mês','pct':'% do Tempo','zone_label':'Zona'},
             category_orders={'zone_label': list(ZONE_LABELS.values())})
fig.add_hline(y=80, line_dash='dash', line_color='green', annotation_text='Meta Z1+Z2 = 80% (Polarizado)')
fig.update_layout(xaxis_tickangle=-45, yaxis_range=[0,100])
fig.show()

---
## 📉 Pilar 4 — Estratégia de Ritmo em Corridas Curtas (5–8km)
> **Fisiologia**: Em treinos curtos e intensos, o erro clássico é o *positive split* — sair forte demais no 1º km e pagar o preço nos seguintes. O **Fator de Saída** mede quanto o 1º km foi mais rápido que a média dos demais: valores > 10% indicam gestão incorreta do esforço, elevando o lactato precocemente e comprometendo o rendimento. Um atleta experiente executa *negative split* (acelera progressivamente) ou ritmo uniforme.

In [31]:
# Corridas curtas (5–8 km): decaimento de pace split a split
q = '''
    SELECT l.activity_id, l.lap_index, l.moving_time, l.distance,
           l.average_heartrate,
           a.start_date_local, a.sport_type, a.name AS activity_name,
           a.distance AS total_dist
    FROM activity_laps l
    JOIN activities a ON l.activity_id = a.id
    WHERE a.distance BETWEEN 5000 AND 8000
      AND a.sport_type IN ('Run','TrailRun')
      AND l.distance > 0 AND l.moving_time > 0
'''
df_laps = pd.read_sql(q, conn)
df_laps['start_date_local'] = pd.to_datetime(df_laps['start_date_local'])

# Filtros dinâmicos
if DATE_START: df_laps = df_laps[df_laps['start_date_local'] >= pd.Timestamp(DATE_START)]
if DATE_END:   df_laps = df_laps[df_laps['start_date_local'] <= pd.Timestamp(DATE_END)]
if SPORT_TYPES: df_laps = df_laps[df_laps['sport_type'].isin(SPORT_TYPES)]

df_laps['pace_min_km'] = (df_laps['moving_time'] / 60) / (df_laps['distance'] / 1000)
df_laps = df_laps[df_laps['pace_min_km'].between(2.5, 15)]

print(f"{df_laps['activity_id'].nunique()} corridas curtas encontradas ({len(df_laps)} splits totais)")

# 10 corridas mais recentes para visualizar a estratégia de ritmo
recentes = df_laps.groupby('activity_id')['start_date_local'].first().nlargest(10).index
df_top = df_laps[df_laps['activity_id'].isin(recentes)].copy()
df_top['label'] = (df_top['start_date_local'].dt.strftime('%Y-%m-%d') + ' (' +
                   (df_top['total_dist']/1000).round(1).astype(str) + 'km)')

fig = px.line(
    df_top, x='lap_index', y='pace_min_km', color='label', markers=True,
    title='Estratégia de Ritmo — Corridas Curtas (5–8km) | 10 mais recentes',
    labels={'lap_index':'Quilômetro','pace_min_km':'Pace (min/km)','label':'Atividade'}
)
fig.update_yaxes(autorange='reversed')
# Linha horizontal: pace médio da atividade (referência de equilíbrio)
pace_medio = df_top.groupby('label')['pace_min_km'].mean()
for label, p in pace_medio.items():
    fig.add_hline(y=p, line_dash='dot', opacity=0.3)
fig.show()

16 corridas curtas encontradas (109 splits totais)


In [32]:
# Fator de Saída: quanto o 1º km foi mais rápido que a média dos demais
# Positivo = atleta saiu forte demais (positive split) | Negativo = acelerou no fim (negative split)
def fator_saida(group):
    if len(group) < 2: return np.nan
    primeiro = group[group['lap_index'] == group['lap_index'].min()]['pace_min_km'].values[0]
    restantes = group[group['lap_index'] > group['lap_index'].min()]['pace_min_km'].mean()
    # Pace menor = mais rápido; se primeiro < restantes, saiu forte (positive split)
    return round((restantes - primeiro) / restantes * 100, 2)

fs = df_laps.groupby('activity_id', group_keys=False).apply(fator_saida).reset_index()
fs.columns = ['activity_id', 'fator_saida_pct']
fs = fs.merge(
    df[['id','start_date_local','name','dist_km']].rename(columns={'id':'activity_id'}),
    on='activity_id'
).dropna().sort_values('start_date_local')

fs['estrategia'] = fs['fator_saida_pct'].apply(
    lambda x: 'Negative Split ✔' if x < -3
              else ('Ritmo Uniforme ≈' if -3 <= x <= 5
              else ('Positive Split leve ⚠ï¸' if 5 < x <= 10
              else 'Positive Split crítico 🚨'))
)
COR = {
    'Negative Split ✔': '#43A047',
    'Ritmo Uniforme ≈': '#26C6DA',
    'Positive Split leve ⚠ï¸': '#FFA726',
    'Positive Split crítico 🚨': '#E53935',
}

fig = px.bar(
    fs, x='start_date_local', y='fator_saida_pct',
    color='estrategia', color_discrete_map=COR,
    hover_data=['name','dist_km'],
    title='Fator de Saída por Corrida Curta (5–8km) — Positive vs Negative Split',
    labels={'start_date_local':'Data','fator_saida_pct':'Fator de Saída (%)','estrategia':'Estratégia'}
)
fig.add_hline(y=10,  line_dash='dash', line_color='red',   annotation_text='Crítico (>10%)')
fig.add_hline(y=5,   line_dash='dash', line_color='orange',annotation_text='Leve (>5%)')
fig.add_hline(y=-3,  line_dash='dash', line_color='green', annotation_text='Negative Split (<-3%)')
fig.add_hline(y=0,   line_color='white', opacity=0.2)
fig.show()

# Resumo estatístico
print(fs['estrategia'].value_counts().to_string())
print(f"Fator médio: {fs['fator_saida_pct'].mean():.1f}% | Mediana: {fs['fator_saida_pct'].median():.1f}%")

estrategia
Ritmo Uniforme ≈            11
Positive Split leve ⚠ï¸     4
Negative Split ✔             1
Fator médio: 2.2% | Mediana: 2.8%


---
## ⛰️ Pilar 5 — Performance vs Altimetria
> **Fisiologia**: A altimetria penaliza o pace de forma não-linear. O gráfico de densidade (streams) revela a curva real de custo energético por grau de inclinação — referência para planejamento de provas de trail e corridas de rua com relevo.

In [33]:
df_elev = df[(df['total_elevation_gain'] > 0) & df['pace_min_km'].between(3, 18)].copy()
df_elev['elev_per_km'] = df_elev['total_elevation_gain'] / df_elev['dist_km']

fig = px.scatter(
    df_elev, x='elev_per_km', y='pace_min_km',
    color='sport_type', size='dist_km', trendline='ols',
    hover_data=['name','dist_km','total_elevation_gain'],
    title='Impacto da Altimetria no Pace (Ganho de Elevação por km vs. Pace)',
    labels={'elev_per_km':'Elevação por km (m/km)','pace_min_km':'Pace (min/km)'}
)
fig.update_yaxes(autorange='reversed')
fig.show()

In [34]:
# Pace vs Inclinação real (streams) — requer load_streams=True no pipeline
n_streams = pd.read_sql('SELECT COUNT(*) as n FROM activity_streams', conn).iloc[0, 0]

if n_streams > 0:
    s = pd.read_sql('''
        SELECT grade_smooth, velocity_smooth, heartrate
        FROM activity_streams
        WHERE velocity_smooth > 0 AND grade_smooth BETWEEN -30 AND 30
        ORDER BY RANDOM() LIMIT 40000
    ''', conn)
    s['pace_min_km'] = (1000 / s['velocity_smooth']) / 60
    s = s[s['pace_min_km'].between(2.5, 18)]

    fig = px.density_heatmap(
        s, x='grade_smooth', y='pace_min_km', nbinsx=40, nbinsy=40,
        title='Pace vs Inclinação do Terreno — Densidade (Streams)',
        labels={'grade_smooth':'Inclinação (%)','pace_min_km':'Pace (min/km)'},
        color_continuous_scale='Viridis'
    )
    fig.update_yaxes(autorange='reversed')
    fig.show()
else:
    print('⚠️ Streams não populados. Execute extract_load.py com load_streams=True para habilitar esta análise.')

⚠️ Streams não populados. Execute extract_load.py com load_streams=True para habilitar esta análise.


---
## 🗺️ Pilar 6 — Mapa Geoespacial (Heatmap de Rotas)
> **Insight**: O mapa de calor revela padrões geográficos de treino — regiões preferidas, repetições de percurso e exposição a terrenos de maior relevo. Útil para correlacionar com picos de lesão por sobrecarga de um tipo de terreno específico.

In [35]:
n_streams = pd.read_sql('SELECT COUNT(*) as n FROM activity_streams WHERE lat IS NOT NULL', conn).iloc[0,0]

if n_streams > 0:
    coords_df = pd.read_sql(
        'SELECT lat, lng FROM activity_streams WHERE lat IS NOT NULL AND lng IS NOT NULL LIMIT 200000', conn)
    coords = list(zip(coords_df['lat'], coords_df['lng']))
    source = f'streams ({n_streams:,} pontos)'
else:
    # Fallback: decodificar polylines comprimidas das atividades
    poly_df = pd.read_sql(
        "SELECT map_summary_polyline FROM activities WHERE map_summary_polyline IS NOT NULL AND map_summary_polyline != ''", conn)
    coords = []
    for encoded in poly_df['map_summary_polyline']:
        try: coords.extend(pl.decode(encoded))
        except: pass
    source = f'polylines ({len(poly_df)} atividades)'

if coords:
    sample = coords if len(coords) <= 100000 else coords[::len(coords)//100000]
    clat = np.mean([c[0] for c in sample[:2000]])
    clng = np.mean([c[1] for c in sample[:2000]])

    m = folium.Map(location=[clat, clng], zoom_start=12, tiles='CartoDB dark_matter')
    HeatMap(sample, radius=8, blur=12, max_zoom=14, min_opacity=0.25).add_to(m)
    folium.LayerControl().add_to(m)

    out = Path('..') / 'data' / 'heatmap_treinos.html'
    m.save(out)
    print(f'🗺️ Mapa gerado via {source}')
    print(f'💾 Salvo em: {out.resolve()}')
    display(m)
else:
    print('⚠️ Nenhuma coordenada encontrada. Verifique se as atividades têm GPS.')

🗺️ Mapa gerado via polylines (230 atividades)
💾 Salvo em: C:\Users\gabri\OneDrive\Belgeler\GitHub\strava-performance-analytics\data\heatmap_treinos.html


---
## 🔢 Pilar 7 — Matriz de Correlação (Pearson)
> **Interpretação**: Correlações fortes (|r| > 0.6) revelam relações causais ou confundidoras entre variáveis. Ex: FC alta correlacionada com distância pode indicar fadiga acumulada; correlação negativa entre temperatura e FC pode sinalizar desidratação no calor.

In [36]:
cols_map = {
    'dist_km':               'Distância (km)',
    'duration_h':            'Duração (h)',
    'total_elevation_gain':  'Elevação (m)',
    'speed_kmh':             'Velocidade (km/h)',
    'average_heartrate':     'FC Média (bpm)',
    'average_temp':          'Temperatura (°C)',
    'cardiac_cost':          'Custo Cardíaco',
    'kilojoules':            'Energia (kJ)',
}
available = {k: v for k, v in cols_map.items() if k in df.columns}
df_c = df[list(available.keys())].dropna().rename(columns=available)

corr = df_c.corr()
fig = px.imshow(
    corr, text_auto='.2f', aspect='auto',
    color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
    title='Matriz de Correlação de Pearson — Variáveis de Performance'
)
fig.update_layout(width=720, height=620)
fig.show()

# Top correlações com FC Média
if 'FC Média (bpm)' in corr.columns:
    top = corr['FC Média (bpm)'].drop('FC Média (bpm)').abs().sort_values(ascending=False)
    print('\n📌 Correlações com FC Média (ordenado por força):')
    for k, v in top.items():
        sinal = '↑' if corr.loc[k,'FC Média (bpm)'] > 0 else '↓'
        print(f'  {sinal} {k:<25} r = {corr.loc[k,"FC Média (bpm)"]:.3f}')


📌 Correlações com FC Média (ordenado por força):
  ↓ Distância (km)            r = nan
  ↓ Duração (h)               r = nan
  ↓ Elevação (m)              r = nan
  ↓ Velocidade (km/h)         r = nan
  ↓ Temperatura (°C)          r = nan
  ↓ Custo Cardíaco            r = nan
  ↓ Energia (kJ)              r = nan
